# Mini MapReduce Word Count (2 Reducers)

This notebook implements the requested workflow:

1. **Input:** Read a folder of text files; each file is one split.
2. **Map:** Emit `(word, 1)` pairs for each word in a file.
3. **Parallel mapping:** Run one mapper per file using a thread pool.
4. **Combine + Partition:** Count words locally in each mapper, then assign each word to one of two reducers.
5. **Shuffle & Sort + Reduce:** Group and sort values by word, sum them, and write one output file per reducer.

The code uses only Python's standard library. Run the cells from top to bottom.

In [1]:
import os
import re
import shutil
import zlib
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

# Configuration
INPUT_DIR = "input"
OUTPUT_DIR = "output"
NUM_REDUCERS = 2


## 1. Create example input files

This cell creates a small `input` folder so the notebook can run immediately. If you already have your own text files, replace these examples with your files in the `input` folder.

In [2]:
# Re-create the sample input folder each time this cell runs.
os.makedirs(INPUT_DIR, exist_ok=True)

sample_files = {
    "file1.txt": "car restaurant car food",
    "file2.txt": "restaurant food car road",
    "file3.txt": "food restaurant road car",
}

for filename, content in sample_files.items():
    with open(os.path.join(INPUT_DIR, filename), "w", encoding="utf-8") as f:
        f.write(content)

print("Input files:")
for filename in sorted(os.listdir(INPUT_DIR)):
    print("-", filename)


Input files:
- file1.txt
- file2.txt
- file3.txt


## 2. Map phase

Each file is one split. A mapper reads one file and emits `(word, 1)` pairs. Words are converted to lowercase, and punctuation is ignored. The mapper also creates a **local combiner** using `Counter`, so repeated words in the same file are summarized before being sent onward.

In [3]:
def mapper(file_path):
    """Process one input split and return local word counts."""
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read().lower()

    # Extract words; punctuation and numbers are not treated as words.
    words = re.findall(r"\\b[a-z]+\\b", text)

    # Map output would be (word, 1) for every word.
    map_pairs = [(word, 1) for word in words]

    # Combiner: add up repeated words locally in this split.
    local_counts = Counter()
    for word, value in map_pairs:
        local_counts[word] += value

    return file_path, map_pairs, local_counts


## 3. Run mappers in parallel

Each input file is submitted as a separate task to `ThreadPoolExecutor`. Results are collected as the tasks finish. The displayed output helps verify that each file was handled by a mapper.

In [4]:
def run_mappers(input_dir):
    file_paths = [
        os.path.join(input_dir, name)
        for name in sorted(os.listdir(input_dir))
        if name.lower().endswith(".txt")
    ]

    if not file_paths:
        raise FileNotFoundError(
            f"No .txt files found in '{input_dir}'. Add text files and run again."
        )

    mapper_results = []
    with ThreadPoolExecutor(max_workers=len(file_paths)) as executor:
        futures = [executor.submit(mapper, path) for path in file_paths]

        for future in as_completed(futures):
            mapper_results.append(future.result())

    # Sort results by filename for predictable display.
    mapper_results.sort(key=lambda result: result[0])
    return mapper_results


mapper_results = run_mappers(INPUT_DIR)

for file_path, map_pairs, local_counts in mapper_results:
    print(f"\\nMapper for {os.path.basename(file_path)}")
    print("Map pairs:", map_pairs)
    print("Local combined counts:", dict(local_counts))


\nMapper for file1.txt
Map pairs: []
Local combined counts: {}
\nMapper for file2.txt
Map pairs: []
Local combined counts: {}
\nMapper for file3.txt
Map pairs: []
Local combined counts: {}


## 4. Combine and partition

The combiner has already summarized each mapper's local word counts. Next, each `(word, local_count)` subtotal is sent to exactly one reducer using a stable hash partition function. A stable hash is used instead of Python's built-in `hash()`, whose results can vary between program runs.

In [5]:
def partition(word):
    """Return a reducer ID from 0 to NUM_REDUCERS - 1."""
    return zlib.crc32(word.encode("utf-8")) % NUM_REDUCERS


def combine_and_partition(mapper_results):
    # One list of (word, subtotal) pairs for each reducer.
    reducer_inputs = [[] for _ in range(NUM_REDUCERS)]

    for file_path, map_pairs, local_counts in mapper_results:
        for word, subtotal in local_counts.items():
            reducer_id = partition(word)
            reducer_inputs[reducer_id].append((word, subtotal))

    return reducer_inputs


reducer_inputs = combine_and_partition(mapper_results)

for reducer_id, pairs in enumerate(reducer_inputs):
    print(f"Reducer {reducer_id} input:", pairs)


Reducer 0 input: []
Reducer 1 input: []


## 5. Shuffle and sort

For each reducer, group all subtotals for the same word into a list, then sort the words alphabetically. This simulates the shuffle-and-sort stage: all values for a word are brought together at the reducer assigned to that word.

In [6]:
def shuffle_and_sort(reducer_inputs):
    shuffled_data = []

    for reducer_id, pairs in enumerate(reducer_inputs):
        grouped = defaultdict(list)

        for word, subtotal in pairs:
            grouped[word].append(subtotal)

        # Sort keys so reducer processing is deterministic and easy to inspect.
        sorted_grouped = {
            word: sorted(grouped[word])
            for word in sorted(grouped)
        }
        shuffled_data.append(sorted_grouped)

    return shuffled_data


shuffled_data = shuffle_and_sort(reducer_inputs)

for reducer_id, grouped_words in enumerate(shuffled_data):
    print(f"\\nReducer {reducer_id} after shuffle and sort:")
    for word, values in grouped_words.items():
        print(f"{word}: {values}")


\nReducer 0 after shuffle and sort:
\nReducer 1 after shuffle and sort:


## 6. Reduce and write output files

Each reducer sums the values for each word assigned to it. The results are written to `output/part-00000.txt` and `output/part-00001.txt`, one file per reducer.

In [7]:
def reduce_and_write(shuffled_data, output_dir):
    # Clear old output so repeated runs don't leave stale reducer files.
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    final_results = []

    for reducer_id, grouped_words in enumerate(shuffled_data):
        output_path = os.path.join(output_dir, f"part-{reducer_id:05d}.txt")
        reducer_result = {}

        with open(output_path, "w", encoding="utf-8") as output_file:
            for word in sorted(grouped_words):
                total = sum(grouped_words[word])
                reducer_result[word] = total
                output_file.write(f"{word}\\t{total}\\n")

        final_results.append(reducer_result)
        print(f"Wrote reducer {reducer_id} output to: {output_path}")

    return final_results


final_results = reduce_and_write(shuffled_data, OUTPUT_DIR)


Wrote reducer 0 output to: output\part-00000.txt
Wrote reducer 1 output to: output\part-00001.txt


## 7. Display the final output files

In [8]:
for filename in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, filename)
    print(f"\\n--- {filename} ---")
    with open(path, "r", encoding="utf-8") as f:
        print(f.read(), end="")


\n--- part-00000.txt ---
\n--- part-00001.txt ---
